In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import lightgbm as lgb
import xgboost as xgb
import joblib
import warnings
import os
warnings.filterwarnings("ignore")

In [2]:
os.makedirs("plots/models", exist_ok=True)
os.makedirs("saved_models", exist_ok=True)
sns.set_theme(style="whitegrid", font_scale=1.1)

In [3]:
train = pd.read_csv("tshirts_train.csv", parse_dates=["week_start"])
val   = pd.read_csv("tshirts_val.csv",   parse_dates=["week_start"])
test  = pd.read_csv("tshirts_test.csv",  parse_dates=["week_start"])
TARGET = "log_sales_volume"  

In [4]:
MACRO_COLS = [
    "eurozone_hicp", "eurozone_unemployment_rate", "eurozone_cci",
    "hicp_lag1m", "unemployment_lag1m", "cci_lag1m",
]

In [5]:
OHE_COLS = [c for c in train.columns if any(
    c.startswith(p) for p in [
        "index_group_name_", "colour_group_name_",
        "graphical_appearance_name_", "perceived_colour_value_name_"
    ]
)]

In [6]:
BASE_FEATURES = [
    "avg_weekly_price", "real_price", "price_vs_median",
    "week_of_year", "month", "quarter",
    "is_spring_summer", "is_sale_season",
    "covid",                    # COVID-19 dummy (1 from Mar 2020)
    "product_age_weeks",
    "sales_lag1", "sales_lag2", "sales_lag4",
    "sales_rolling4_mean", "sales_rolling4_std",
] + OHE_COLS


In [7]:
MACRO_FEATURES = BASE_FEATURES + MACRO_COLS

In [8]:
FEATURE_SETS = {
    "Baseline (no macro)":       BASE_FEATURES,
    "Macro-enriched":            MACRO_FEATURES,
}

In [9]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

In [10]:
def nz_mape(y_true, y_pred):
    """MAPE computed only on weeks with non-zero actual sales.
    Using all rows (incl. 82% zeros) inflates MAPE to millions of percent."""
    mask = y_true > 0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

In [11]:
def evaluate(model, X, y_log, label=""):
    """Predict, inverse-transform, compute all metrics."""
    pred_log = model.predict(X)
    pred_log = np.clip(pred_log, 0, None)         # sales can't be negative
    y_raw    = np.expm1(y_log)                    # back to original scale
    pred_raw = np.expm1(pred_log)
    return {
        "label":    label,
        "RMSE":     rmse(y_raw, pred_raw),
        "MAE":      mean_absolute_error(y_raw, pred_raw),
        "NZ-MAPE":  nz_mape(y_raw, pred_raw),
        "pred_log": pred_log,
        "y_raw":    y_raw,
        "pred_raw": pred_raw,
    }

In [12]:
def get_models():
    return {
        "LightGBM": lgb.LGBMRegressor(
            n_estimators=500, learning_rate=0.05, num_leaves=63,
            min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
            random_state=42, verbose=-1
        ),
        "XGBoost": xgb.XGBRegressor(
            n_estimators=500, learning_rate=0.05, max_depth=6,
            subsample=0.8, colsample_bytree=0.8,
            random_state=42, verbosity=0, tree_method="hist"
        ),
        "RandomForest": RandomForestRegressor(
            n_estimators=200, max_depth=12, min_samples_leaf=5,
            n_jobs=-1, random_state=42
        ),
    }

In [13]:
all_results = []
trained_models = {}

In [14]:
for fs_name, features in FEATURE_SETS.items():
    # Filter to columns that actually exist in the data
    features = [f for f in features if f in train.columns]
 
    X_train = train[features]; y_train = train[TARGET]
    X_val   = val[features];   y_val   = val[TARGET]
    X_test  = test[features];  y_test  = test[TARGET]
 
    print(f"\n{'='*60}")
    print(f"Feature set: {fs_name}  ({len(features)} features)")
    print(f"{'='*60}")
 
    for model_name, model in get_models().items():
        print(f"  Training {model_name}...", end=" ", flush=True)
 
        # LightGBM: use early stopping on val set
        if model_name == "LightGBM":
            model.fit(
                X_train, y_train,
                eval_set=[(X_val, y_val)],
                callbacks=[lgb.early_stopping(50, verbose=False),
                           lgb.log_evaluation(period=-1)]
            )
        else:
            model.fit(X_train, y_train)
 
        val_metrics  = evaluate(model, X_val,  y_val,  label="val")
        test_metrics = evaluate(model, X_test, y_test, label="test")
 
        print(f"Val RMSE={val_metrics['RMSE']:.3f}  Test RMSE={test_metrics['RMSE']:.3f}")
 
        for split, m in [("val", val_metrics), ("test", test_metrics)]:
            all_results.append({
                "feature_set": fs_name,
                "model":       model_name,
                "split":       split,
                "RMSE":        round(m["RMSE"], 4),
                "MAE":         round(m["MAE"],  4),
                "NZ-MAPE (%)": round(m["NZ-MAPE"], 2),
            })
 
        # Save model
        key = f"{fs_name}__{model_name}"
        trained_models[key] = {
            "model": model, "features": features,
            "val_metrics": val_metrics, "test_metrics": test_metrics
        }
        joblib.dump(model, f"saved_models/{model_name}_{fs_name[:7].replace(' ', '_')}.pkl")

In [15]:
results_df = pd.DataFrame(all_results)
results_df.to_csv("model_results.csv", index=False)

In [16]:
print("\n\nFULL RESULTS TABLE")
print("=" * 75)
print(results_df.to_string(index=False))



FULL RESULTS TABLE
        feature_set        model split   RMSE    MAE   MAPE (%)
Baseline (no macro)     LightGBM   val 5.2051 0.9669 3539753.48
Baseline (no macro)     LightGBM  test 6.2436 1.0702 3285409.10
Baseline (no macro)      XGBoost   val 5.2765 0.9757 3443519.64
Baseline (no macro)      XGBoost  test 6.3489 1.0879 3737852.83
Baseline (no macro) RandomForest   val 5.2160 0.9941 4078733.91
Baseline (no macro) RandomForest  test 6.2333 1.0911 4119398.59
     Macro-enriched     LightGBM   val 5.2255 0.9549 2398457.58
     Macro-enriched     LightGBM  test 6.2789 1.0668 3113333.81
     Macro-enriched      XGBoost   val 5.3153 0.9762 3183877.86
     Macro-enriched      XGBoost  test 6.4445 1.1068 4141947.61
     Macro-enriched RandomForest   val 5.2179 0.9829 2998551.98
     Macro-enriched RandomForest  test 6.2418 1.0900 3961659.76


In [17]:
for split in ["val", "test"]:
    subset = results_df[results_df["split"] == split].copy()
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(subset["model"].unique()))
    width = 0.35
    models_unique = subset["model"].unique()
    fs_unique     = subset["feature_set"].unique()
 
    for i, fs in enumerate(fs_unique):
        vals = subset[subset["feature_set"] == fs].set_index("model")["RMSE"]
        vals = vals.reindex(models_unique)
        offset = (i - 0.5) * width
        bars = ax.bar(x + offset, vals, width, label=fs,
                      color=["steelblue", "coral"][i], edgecolor="white", alpha=0.85)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f"{v:.3f}", ha="center", va="bottom", fontsize=9)
 
    ax.set_xticks(x); ax.set_xticklabels(models_unique)
    ax.set_ylabel("RMSE (original scale)")
    ax.set_title(f"Model RMSE Comparison — {split.upper()} set\n(Lower is better)",
                 fontweight="bold")
    ax.legend()
    plt.tight_layout()
    plt.savefig(f"plots/models/RMSE_comparison_{split}.png")
    plt.close()
    print(f"Saved: RMSE_comparison_{split}.png")

Saved: RMSE_comparison_val.png
Saved: RMSE_comparison_test.png


In [18]:
improvements = []
for model_name in results_df["model"].unique():
    for split in ["val", "test"]:
        sub = results_df[(results_df["model"] == model_name) & (results_df["split"] == split)]
        base = sub[sub["feature_set"].str.startswith("Baseline")]["RMSE"].values
        macro = sub[sub["feature_set"].str.startswith("Macro")]["RMSE"].values
        if len(base) and len(macro):
            improvement_pct = (base[0] - macro[0]) / base[0] * 100
            improvements.append({
                "model": model_name, "split": split,
                "RMSE_baseline": base[0], "RMSE_macro": macro[0],
                "improvement_%": round(improvement_pct, 2)
            })

In [19]:
imp_df = pd.DataFrame(improvements)
print("\nIMPROVEMENT FROM ADDING MACRO FEATURES (RMSE reduction %):")
print(imp_df.to_string(index=False))
 
fig, ax = plt.subplots(figsize=(9, 5))
imp_test = imp_df[imp_df["split"] == "test"]
colors = ["mediumseagreen" if v > 0 else "tomato" for v in imp_test["improvement_%"]]
bars = ax.bar(imp_test["model"], imp_test["improvement_%"],
              color=colors, edgecolor="white", alpha=0.85)
for bar, v in zip(bars, imp_test["improvement_%"]):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + (0.1 if v >= 0 else -0.3),
            f"{v:+.1f}%", ha="center", va="bottom", fontweight="bold")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("RMSE Reduction (%) when adding macro features")
ax.set_title("Does Adding Macroeconomic Features Improve Forecasting?\n(Test set — positive = improvement)",
             fontweight="bold")
plt.tight_layout()
plt.savefig("plots/models/macro_improvement.png")
plt.close()
print("Saved: macro_improvement.png")


IMPROVEMENT FROM ADDING MACRO FEATURES (RMSE reduction %):
       model split  RMSE_baseline  RMSE_macro  improvement_%
    LightGBM   val         5.2051      5.2255          -0.39
    LightGBM  test         6.2436      6.2789          -0.57
     XGBoost   val         5.2765      5.3153          -0.74
     XGBoost  test         6.3489      6.4445          -1.51
RandomForest   val         5.2160      5.2179          -0.04
RandomForest  test         6.2333      6.2418          -0.14
Saved: macro_improvement.png


In [20]:
test_rows = results_df[
    (results_df["split"] == "test") &
    (results_df["feature_set"].str.startswith("Macro"))
]
best_row = test_rows.loc[test_rows["RMSE"].idxmin()]
best_key = f"{best_row['feature_set']}__{best_row['model']}"
print(f"\nBest model: {best_row['model']} ({best_row['feature_set']})  Test RMSE={best_row['RMSE']:.4f}")


Best model: RandomForest (Macro-enriched)  Test RMSE=6.2418


In [21]:
best_entry   = trained_models[best_key]
best_model   = best_entry["model"]
best_feats   = best_entry["features"]
test_metrics = best_entry["test_metrics"]

In [22]:
# Aggregate predictions by week for a readable time-series comparison
test["pred_raw"] = test_metrics["pred_raw"]
test["y_raw"]    = test_metrics["y_raw"]
 
weekly_pred = test.groupby("week_start").agg(
    actual=("y_raw",    "sum"),
    predicted=("pred_raw", "sum")
).reset_index()
 
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(weekly_pred["week_start"], weekly_pred["actual"],
        label="Actual", color="steelblue", linewidth=2)
ax.plot(weekly_pred["week_start"], weekly_pred["predicted"],
        label="Predicted", color="coral", linewidth=2, linestyle="--")
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(plt.matplotlib.dates.MonthLocator(interval=1))
plt.xticks(rotation=30)
ax.set_title(f"Actual vs. Predicted Weekly Sales — {best_row['model']} (Macro-Enriched, Test Set)",
             fontweight="bold")
ax.set_ylabel("Total Weekly Sales"); ax.legend()
plt.tight_layout()
plt.savefig("plots/models/actual_vs_predicted.png")
plt.close()
print("Saved: actual_vs_predicted.png")

Saved: actual_vs_predicted.png


In [23]:
# Save text summary
summary_lines = [
    "MODEL EVALUATION SUMMARY",
    "=" * 65,
    results_df.to_string(index=False),
    "\nIMPROVEMENT FROM MACRO FEATURES:",
    imp_df.to_string(index=False),
    f"\nBest model: {best_row['model']} ({best_row['feature_set']})",
    f"Test RMSE: {best_row['RMSE']}  MAE: {best_row['MAE']}  NZ-MAPE: {best_row['NZ-MAPE (%)']}%",
]
with open("model_results.txt", "w") as f:
    f.write("\n".join(summary_lines))

print("\nModel training complete. Results saved to model_results.csv and model_results.txt")